# t-SNE & UMAP

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/pca-dimensionality/02-t-sne-and-umap

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — non-linear maps for visualization

PCA is linear, so it can't unfold data that lives on a **curved manifold** — digit images, for
instance, cluster in ways a flat projection smears together. **t-SNE** and **UMAP** are non-linear
dimensionality-reduction methods built for **visualization**: they convert high-dimensional distances
into neighbor *probabilities* and then lay out points in 2-D so that **nearby points stay nearby**.
t-SNE's key knob is **perplexity** (roughly, the effective number of neighbors each point considers),
set by binary-searching a per-point bandwidth `σ`. **UMAP** does something similar but faster and with
better **global** structure. The catch: these are *visualization* tools, and their output must be read
with care. We validate the perplexity math and compare against `sklearn`.

## PCA vs t-SNE

PCA finds linear projections that maximize variance. t-SNE preserves local neighborhood structure.

In [ ]:
from sklearn.decomposition import PCA

digits = load_digits()
X, y = digits.data, digits.target

X_pca = PCA(n_components=2).fit_transform(X)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=5, alpha=0.7)
axes[0].set_title('PCA — Linear Projection', color='white', fontsize=12)

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='tab10', s=5, alpha=0.7)
axes[1].set_title('t-SNE — Preserves Local Structure', color='white', fontsize=12)

for ax in axes:
    ax.axis('off')
plt.suptitle('MNIST Digits: PCA vs t-SNE', color='white', fontsize=13)
plt.tight_layout()
plt.show()

**What to notice:** on the digits, **PCA** (linear) overlaps the classes into a smear, while
**t-SNE** pulls them apart into ten clean clusters — because it optimizes to preserve *local*
neighborhoods rather than global variance. For *seeing* cluster structure, non-linear methods win; for
*compression* with meaningful axes, PCA is still the tool.

## Effect of perplexity in t-SNE

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, perp in zip(axes, [5, 30, 100]):
    X_embedded = TSNE(n_components=2, perplexity=perp, random_state=42).fit_transform(X)
    ax.scatter(X_embedded[:, 0], X_embedded[:, 1], c=y, cmap='tab10', s=5, alpha=0.7)
    ax.set_title(f'Perplexity = {perp}', color='white', fontsize=11)
    ax.axis('off')
plt.suptitle('t-SNE: Perplexity Controls Local vs Global Structure', color='white', fontsize=13)
plt.tight_layout()
plt.show()

**What to notice:** **perplexity** balances local vs global focus — too **small** and t-SNE
fragments clusters into little islands, too **large** and it blurs distinct clusters together. It's
roughly "how many neighbors each point pays attention to," and 5–50 is the usual range. There's no
single correct value; try a few.

## What perplexity actually does: tuning sigma

Perplexity is $2^{H(P_i)}$ where $H$ is the Shannon entropy (bits) of point $i$'s conditional neighbor distribution $p_{j|i}\propto e^{-d_{ij}^2/2\sigma_i^2}$. t-SNE binary-searches each $\sigma_i$ so the perplexity matches the target — i.e. so each point has a chosen *effective number of neighbors*. Below we reproduce that search by hand and confirm the identities from the lesson.

In [ ]:
import numpy as np

def perplexity_of(p):
    """Perplexity = 2^H, H = Shannon entropy (bits) of distribution p."""
    p = p[p > 0]
    H = -np.sum(p * np.log2(p))
    return 2.0 ** H

# Identities from the lesson:
print('uniform over 4 neighbors -> perplexity =', perplexity_of(np.full(4, 0.25)))   # 4.0
print('peaked (0.97,0.01,0.01,0.01) -> perplexity =',
      round(perplexity_of(np.array([0.97, 0.01, 0.01, 0.01])), 3))                    # ~1.18

# Conditional p_{j|i} from squared distances with bandwidth sigma
def conditional_p(d2, sigma):
    w = np.exp(-d2 / (2 * sigma**2))
    return w / w.sum()

def sigma_for_perplexity(d2, target, lo=1e-3, hi=1e3, iters=60):
    """Binary-search sigma so perplexity(p_{j|i}) == target (t-SNE's exact procedure)."""
    for _ in range(iters):
        mid = (lo + hi) / 2
        perp = perplexity_of(conditional_p(d2, mid))
        if perp < target:   # too peaked -> widen sigma
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

# One point with 10 neighbors at increasing distances
rng = np.random.default_rng(0)
d2 = np.sort(rng.uniform(0.5, 30, size=10))
for target in [2, 5, 8]:
    s = sigma_for_perplexity(d2, target)
    p = conditional_p(d2, s)
    print(f'target perp {target}: sigma={s:.3f}, achieved perp={perplexity_of(p):.3f}, '
          f'top-3 p={np.round(np.sort(p)[::-1][:3], 3)}')
print('\nHigher target perplexity -> larger sigma -> probability spread over MORE neighbors.')


**What to notice:** under the hood, t-SNE **binary-searches a bandwidth `σ` per point** so that
each point's neighbor distribution has the *target perplexity*. Dense regions get a small `σ`, sparse
regions a large one — an adaptive scale that's why t-SNE handles varying density better than a fixed
radius.

## The library way — the perplexity–entropy identity, and `sklearn`

t-SNE and UMAP *are* the libraries (`sklearn.manifold.TSNE`, `umap-learn`). The one piece worth
verifying from scratch is the definition of perplexity: it's `2` raised to the **entropy** of the
neighbor distribution, so a distribution spread uniformly over `k` neighbors has perplexity exactly
`k` — literally "effective number of neighbors."

In [ ]:
# perplexity = 2^(entropy of the conditional neighbor distribution)
def perplexity_of(p):
    p = p[p > 0]
    return 2 ** (-np.sum(p * np.log2(p)))

# a distribution uniform over k neighbors has perplexity exactly k
for k in [2, 8, 30]:
    u = np.ones(k) / k
    print(f'uniform over {k:>2} neighbors -> perplexity {perplexity_of(u):.1f}')
    assert np.isclose(perplexity_of(u), k), "uniform over k -> perplexity k"

# a peaked distribution has LOW perplexity (few effective neighbors)
peaked = np.array([0.9, 0.05, 0.03, 0.02])
print(f'peaked distribution      -> perplexity {perplexity_of(peaked):.2f}  (few effective neighbors)')
print('\nperplexity = 2^entropy = effective number of neighbors ✓')

**What to notice:** a distribution spread evenly over `k` neighbors has perplexity exactly `k`, while
a peaked one has perplexity near 1 — confirming perplexity is the *effective neighbor count*. Setting it
tells t-SNE how big a neighborhood to preserve, which the per-point `σ` search then enforces.

## UMAP: faster, preserves more global structure

UMAP often beats t-SNE on speed and keeps more of the global layout. Key knobs: `n_neighbors` (local vs global) and `min_dist` (cluster tightness).

In [ ]:
from sklearn.datasets import load_digits
X, y = load_digits(return_X_y=True)
try:
    import umap
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X)
    plt.scatter(emb[:, 0], emb[:, 1], c=y, cmap='Spectral', s=6)
    plt.title('UMAP of digits'); plt.colorbar(label='digit'); plt.show()
except ImportError:
    print('pip install umap-learn to run this cell')

**What to notice:** **UMAP** produces similarly clean clusters much **faster** than t-SNE and tends
to preserve more **global** structure (the relative arrangement of clusters is more meaningful). It's
the modern default for large datasets. (This cell needs `umap-learn`; in Colab, `pip install umap-learn`
first — it prints a hint if missing.)

## Gotchas & tradeoffs

- **These are visualization tools, not feature extractors.** Distances and densities in a t-SNE/UMAP
  plot are **not** faithful — don't feed the 2-D coordinates into a downstream model or read exact
  distances off them.
- **Non-deterministic.** Different random seeds give different layouts; fix the seed and run a few.
- **Cluster sizes and gaps can be misleading.** t-SNE inflates dense clusters and equalizes densities,
  so a big blob isn't necessarily a big class, and gap widths aren't distances.
- **They can manufacture clusters.** With the wrong perplexity, t-SNE will split uniform noise into
  fake clumps — always cross-check with the data / a second method.

In [ ]:
# t-SNE/UMAP DISTORT distances: they preserve neighbor *ranks*, not metric distances.
# Illustration: a peaked vs uniform neighbor distribution have very different perplexities
# even if built from the same raw distances -> the embedding's geometry is not the data's geometry.
raw_dists = np.array([1.0, 1.1, 1.2, 5.0])            # one far point, three close
for sigma in [0.5, 3.0]:
    aff = np.exp(-raw_dists**2 / (2 * sigma**2)); aff /= aff.sum()
    print(f'sigma={sigma}: neighbor probs {aff.round(3)}  perplexity {perplexity_of(aff):.2f}')
print('\n-> the SAME distances yield different neighborhoods depending on sigma/perplexity;')
print('   the 2-D layout reflects those choices, not raw distances -> don\'t over-read the geometry')

**What to notice:** the same raw distances produce very different neighbor probabilities (and
perplexities) depending on `σ` — so the embedding's geometry encodes the *algorithm's choices*, not the
data's true distances. This is why t-SNE/UMAP plots are for building intuition about *neighborhoods*,
never for measuring distances or feeding a downstream model.

## Key takeaways

- **t-SNE** and **UMAP** are non-linear methods for **visualizing** high-dim data in 2-D.
- They preserve **local** neighborhoods; distances/sizes between clusters aren't literal.
- t-SNE's **perplexity** and UMAP's **n_neighbors** balance local vs global structure.
- Use PCA to denoise first; never use these embeddings as features for a downstream model.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Perplexity is 2^entropy

t-SNE's perplexity knob is the **effective number of neighbors** of a point's affinity distribution:

$$\text{Perp}(p) = 2^{H(p)}, \qquad H(p) = -\sum_i p_i \log_2 p_i$$

Implement it. Uniform attention over $k$ neighbors gives perplexity exactly $k$; all attention on one neighbor gives 1.

In [ ]:
def perplexity(p):
    """Perplexity (2^entropy) of a discrete distribution."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]

    # TODO(you): the entropy in bits
    H = ...

    # TODO(you): 2 ** H
    return ...

In [ ]:
# Checks — run me
assert abs(perplexity([0.25] * 4) - 4) < 1e-9, "uniform over 4 neighbors -> perplexity 4"
assert abs(perplexity([1.0, 0.0, 0.0]) - 1) < 1e-9, "all mass on one neighbor -> perplexity 1"
assert 1 < perplexity([0.7, 0.2, 0.1]) < 3, "in between: an 'effective number of neighbors'"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def perplexity(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    H = -np.sum(p * np.log2(p))
    return float(2 ** H)
```

</details>

### Exercise 2 — Binary-searching σ

This is the search t-SNE runs **for every point**: perplexity increases monotonically with $\sigma$ (wider Gaussian → attention spreads over more neighbors), so a binary search finds the $\sigma$ that hits the target. Implement the update rule: if the current perplexity is *below* target, $\sigma$ is too small — move the lower bound up; otherwise move the upper bound down.

In [ ]:
def affinities(dists, sigma):
    """Gaussian affinities over a point's neighbor distances, normalized."""
    dists = np.asarray(dists, dtype=float)
    w = np.exp(-dists ** 2 / (2 * sigma ** 2))
    return w / w.sum()


def find_sigma(dists, target_perp, lo=1e-3, hi=1e3, iters=60):
    """Binary-search the sigma whose affinities hit the target perplexity."""
    for _ in range(iters):
        mid = (lo + hi) / 2

        # TODO(you): if perplexity(affinities(dists, mid)) is below target,
        # sigma is too small -> lo = mid; otherwise hi = mid
        ...

    return (lo + hi) / 2

In [ ]:
# Checks — run me
d = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

sigma = find_sigma(d, 3.0)
assert abs(perplexity(affinities(d, sigma)) - 3.0) < 1e-3, "binary search hits the target perplexity"

assert find_sigma(d, 1.5) < find_sigma(d, 4.5), "higher target perplexity needs a wider Gaussian"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def find_sigma(dists, target_perp, lo=1e-3, hi=1e3, iters=60):
    for _ in range(iters):
        mid = (lo + hi) / 2
        if perplexity(affinities(dists, mid)) < target_perp:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2
```

</details>